# 00 · Start here — anatomy of a calibration node

<div style="background: rgba(128, 128, 128, 0.1); border-left: 4px solid rgba(128, 128, 128, 0.6); padding: 8px 14px; border-radius: 6px;">

A *node* is one calibration experiment. This notebook takes the real
`calibrations/02a_resonator_spectroscopy.py` node and walks through it **one cell per part**.

The stack, top to bottom:

| | Package | Role |
|---|---|---|
| 💎 | **QUA** (`qm.qua`) — [docs](https://docs.quantum-machines.co/latest/) | The pulse-level language that runs on the OPX. *This is where the experiment actually happens.* |
| 🧩 | **QUAM** — [docs](https://qua-platform.github.io/quam/) | The Python object model of your quantum machine (qubits, resonators, pulses). |
| 🧩 | **quam-builder** — [docs](https://github.com/qua-platform/quam-builder) | Builds QUAM classes for a concrete QPU architecture. |
| 🧩 | **QUAlibrate** — [docs](https://qua-platform.github.io/qualibrate/) | Wraps an experiment into a node: parameters, run-actions, results, saving. |
| 🧩 | **qualibration-libs** — [docs](https://github.com/qua-platform/qualibration-libs) | Reusable helpers: qubit selection, data fetching, simulation. |

Callouts marked 💎 are **QUA** — the gem. Callouts marked 🧩 are the supporting packages.

</div>

<div style="background: rgba(230, 160, 0, 0.12); border-left: 4px solid rgba(230, 160, 0, 0.8); padding: 8px 14px; border-radius: 6px;">

Run the cells top to bottom. The execute step (§7) runs on the OPX, so it needs a free machine; set `simulate = True` in §5 only if you have a direct QOP connection (the simulator is not available on IQCC cloud backends).

</div>

## 1 · Imports
Grouped by package so you can see who provides what.

<div style="background: rgba(56, 132, 255, 0.10); border-left: 4px solid rgba(56, 132, 255, 0.75); padding: 8px 14px; border-radius: 6px;">

💎 **QUA** — the heart of the stack. `from qm.qua import *` brings in `program`, `declare`, `for_`, `align`, `save`, `stream_processing` … everything used to write the pulse sequence. [QUA docs](https://docs.quantum-machines.co/latest/)

</div>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from dataclasses import asdict

# ── 💎 QUA ── the pulse-level language
from qm.qua import *

# ── py-qua-tools ── small helpers around QUA (loops, units, progress bar, multi-user sessions)
from qualang_tools.loops import from_array
from qualang_tools.multi_user import qm_session
from qualang_tools.results import progress_counter
from qualang_tools.units import unit

# ── 🧩 QUAlibrate ── the node framework
from qualibrate import QualibrationNode

# ── 🧩 quam-builder ── `Quam` resolves to the architecture class for your QPU
from config.quam_builder import Quam

# ── this repo ── the analysis/plotting for this particular experiment
from calibration_utils.resonator_spectroscopy import (
    Parameters,
    process_raw_dataset,
    fit_raw_data,
    log_fitted_results,
    plot_raw_amplitude_with_fit,
    plot_raw_phase,
)

# ── 🧩 qualibration-libs ── shared helpers
from qualibration_libs.parameters import get_qubits
from qualibration_libs.runtime import simulate_and_plot
from qualibration_libs.data import XarrayDataFetcher

## 2 · Locate the QUAM state
`state.json` and `wiring.json` describe your machine. Make sure they are present
(in a workstation checkout `install.py` fetches them; in Learn `learn_setup.py` does).

In [ ]:
from pathlib import Path

# Everything below comes from the active qualibrate config: the same source the nodes use at runtime.
from qualibrate.core.config.resolvers import get_quam_state_path
from qualibrate_config.resolvers import get_qualibrate_config, get_qualibrate_config_path

_config = get_qualibrate_config(get_qualibrate_config_path())
BACKEND = _config.project
_configured = get_quam_state_path(_config)
if not _configured:
    raise RuntimeError("No QUAM state path configured. Run install.py (workstation) or learn_setup.py (Learn) first.")
# install.py / learn_setup.py store an absolute path; a relative one (hand-run `qualibrate config`)
# means "relative to the checkout root" (the kernel's working directory).
STATE_PATH = Path(_configured)
if not STATE_PATH.is_absolute():
    STATE_PATH = Path.cwd() / STATE_PATH
print(f"backend: {BACKEND}\nstate:   {STATE_PATH}")


## 3 · Load the machine

<div style="background: rgba(46, 160, 67, 0.10); border-left: 4px solid rgba(46, 160, 67, 0.75); padding: 8px 14px; border-radius: 6px;">

🧩 **QUAM** — [docs](https://qua-platform.github.io/quam/) · **quam-builder** — [docs](https://github.com/qua-platform/quam-builder)
`Quam.load(...)` reads the state and returns a Python object tree: `machine.qubits["q1"].resonator`, `.xy`, `.z` …
quam-builder defines the classes in that tree for your QPU architecture.

</div>

In [ ]:
machine = Quam.load(str(STATE_PATH))
print("qubits:", list(machine.qubits))
first = next(iter(machine.qubits.values()))
print("resonator of", first.name, "-> intermediate frequency:", first.resonator.intermediate_frequency)

## 4 · Create the node

<div style="background: rgba(46, 160, 67, 0.10); border-left: 4px solid rgba(46, 160, 67, 0.75); padding: 8px 14px; border-radius: 6px;">

🧩 **QUAlibrate** — [docs](https://qua-platform.github.io/qualibrate/)
`QualibrationNode[Parameters, Quam]` ties together a **name**, a **description**, typed **parameters**
and the **machine**. The two type arguments only give you editor type hints.
The node is named `00_start_here_resonator_spectroscopy` so it can't clash with the real `02a` node.

</div>

In [ ]:
description = """
1D RESONATOR SPECTROSCOPY
Sweep the readout frequency, demodulate I/Q, and fit the resonator resonance.
State update: qubit.resonator.f_01 and qubit.resonator.RF_frequency
"""

node = QualibrationNode[Parameters, Quam](
    name="00_start_here_resonator_spectroscopy",
    description=description,
    parameters=Parameters(),
    machine=Quam.load(str(STATE_PATH)),
)

## 5 · Parameters

<div style="background: rgba(46, 160, 67, 0.10); border-left: 4px solid rgba(46, 160, 67, 0.75); padding: 8px 14px; border-radius: 6px;">

🧩 **QUAlibrate** — `node.parameters` is what the GUI exposes and what graphs override.
🧩 **qualibration-libs** — `Parameters` is composed from mixins in `qualibration_libs.parameters`
(see `calibration_utils/resonator_spectroscopy/parameters.py`).

</div>

`@node.run_action` turns a function into a named step of the node. Here it runs immediately; inside the
QUAlibrate GUI or a graph the same functions are scheduled by the framework, and `skip_if=` decides which ones run.

In [ ]:
@node.run_action(skip_if=node.modes.external)
def custom_param(node: QualibrationNode[Parameters, Quam]):
    """Local overrides for notebook / IDE use. Ignored when run from the GUI or a graph."""
    node.parameters.qubits = [next(iter(node.machine.qubits))]  # just the first qubit
    node.parameters.simulate = False                             # True: OPX simulator (direct QOP only, not IQCC cloud)
    node.parameters.num_shots = 100

print(node.parameters)

## 6 · The QUA program
This is the centerpiece, in two cells: **sweep axes → pulse sequence + stream processing**.

### 6a · Sweep axes (plain Python / NumPy)
No QUA yet — we decide *which frequencies* to measure and name the axes for the resulting dataset.

In [ ]:
@node.run_action(skip_if=node.parameters.load_data_id is not None)
def define_sweep(node: QualibrationNode[Parameters, Quam]):
    u = unit(coerce_to_integer=True)  # py-qua-tools: MHz, us, ns ... as plain numbers
    node.namespace["qubits"] = qubits = get_qubits(node)  # qualibration-libs: resolve parameters.qubits
    span = node.parameters.frequency_span_in_mhz * u.MHz
    step = node.parameters.frequency_step_in_mhz * u.MHz
    dfs = np.arange(-span / 2, +span / 2, step)
    node.namespace["dfs"] = dfs
    node.namespace["sweep_axes"] = {
        "qubit": xr.DataArray(qubits.get_names()),
        "detuning": xr.DataArray(dfs, attrs={"long_name": "readout frequency", "units": "Hz"}),
    }
    print(f"{len(dfs)} frequency points for {len(qubits)} qubit(s)")

### 6b · The pulse sequence and stream processing — QUA

<div style="background: rgba(56, 132, 255, 0.10); border-left: 4px solid rgba(56, 132, 255, 0.75); padding: 8px 14px; border-radius: 6px;">

💎 **QUA** — the heart of the stack. Everything inside `with program()` is **QUA**. It is compiled and executed *on the OPX*, in real time, not in Python. `declare` creates variables that live on the hardware, `for_` is a hardware loop, `save` streams a value back to the host. [QUA docs](https://docs.quantum-machines.co/latest/)

</div>

In [ ]:
@node.run_action(skip_if=node.parameters.load_data_id is not None)
def create_qua_program(node: QualibrationNode[Parameters, Quam]):
    u = unit(coerce_to_integer=True)
    qubits, dfs = node.namespace["qubits"], node.namespace["dfs"]
    num_qubits = len(qubits)
    n_avg = node.parameters.num_shots

    # ═══════════════════════ 💎 QUA PROGRAM STARTS HERE ═══════════════════════
    with program() as node.namespace["qua_program"]:
        # 🧩 QUAM helper: declares I, Q, n QUA variables + their streams, one per qubit
        I, I_st, Q, Q_st, n, n_st = node.machine.declare_qua_variables()
        df = declare(int)                               # 💎 a variable living on the OPX

        for multiplexed_qubits in qubits.batch():       # python-side: groups of qubits measured together
            for qubit in multiplexed_qubits.values():
                node.machine.initialize_qpu(target=qubit)   # 🧩 QUAM: set flux points etc.
            align()                                     # 💎 synchronise all channels
            with for_(n, 0, n < n_avg, n + 1):          # 💎 hardware averaging loop
                save(n, n_st)                           # 💎 stream the iteration counter (progress bar)
                with for_(*from_array(df, dfs)):        # 💎 hardware loop over frequency (from_array: py-qua-tools)
                    for i, qubit in multiplexed_qubits.items():
                        rr = qubit.resonator            # 🧩 QUAM object: a channel with its pulses and frequencies
                        rr.update_frequency(df + rr.intermediate_frequency)   # 💎 QUA update_frequency, driven by QUAM
                        rr.measure("readout", qua_vars=(I[i], Q[i]))          # 💎 play readout pulse + demodulate
                        rr.wait(rr.depletion_time * u.ns)                     # 💎 let the resonator ring down
                        save(I[i], I_st[i])             # 💎 send I to the stream
                        save(Q[i], Q_st[i])             # 💎 send Q to the stream
                    align()

        # 💎 stream_processing() also runs on the OPX: buffer one sweep and average over shots
        # before anything is sent back to Python.
        with stream_processing():
            n_st.save("n")
            for i in range(num_qubits):
                I_st[i].buffer(len(dfs)).average().save(f"I{i + 1}")
                Q_st[i].buffer(len(dfs)).average().save(f"Q{i + 1}")
    # ═══════════════════════ 💎 QUA PROGRAM ENDS HERE ═════════════════════════

## 7 · Simulate or execute

<div style="background: rgba(56, 132, 255, 0.10); border-left: 4px solid rgba(56, 132, 255, 0.75); padding: 8px 14px; border-radius: 6px;">

💎 **QUA** — [docs](https://docs.quantum-machines.co/latest/) · `node.machine.connect()` opens a `QuantumMachinesManager`; `generate_config()` turns the QUAM
tree into the **QUA config** (the OPX's hardware description); `qm.execute(program)` sends the program to the OPX.
🧩 **qualibration-libs** — `simulate_and_plot`, `XarrayDataFetcher` · **py-qua-tools** — `qm_session`, `progress_counter`.

</div>

With `simulate = True` the program runs on the OPX *simulator* and you get the output waveforms back — a safe first look, but only with a direct QOP connection (not on IQCC cloud backends).

In [ ]:
@node.run_action(skip_if=node.parameters.load_data_id is not None or not node.parameters.simulate)
def simulate_qua_program(node: QualibrationNode[Parameters, Quam]):
    qmm = node.machine.connect()                    # 💎 QuantumMachinesManager
    config = node.machine.generate_config()         # 🧩 QUAM -> 💎 QUA config
    samples, fig, wf_report = simulate_and_plot(qmm, config, node.namespace["qua_program"], node.parameters)
    node.results["simulation"] = {"figure": fig, "wf_report": wf_report, "samples": samples}

In [ ]:
@node.run_action(skip_if=node.parameters.load_data_id is not None or node.parameters.simulate)
def execute_qua_program(node: QualibrationNode[Parameters, Quam]):
    qmm = node.machine.connect()
    config = node.machine.generate_config()
    with qm_session(qmm, config, timeout=node.parameters.timeout) as qm:     # py-qua-tools: wait for a free machine
        node.namespace["job"] = job = qm.execute(node.namespace["qua_program"])   # 💎 run on the OPX
        data_fetcher = XarrayDataFetcher(job, node.namespace["sweep_axes"])        # 🧩 qualibration-libs: stream -> xarray
        for dataset in data_fetcher:
            progress_counter(data_fetcher.get("n", 0), node.parameters.num_shots, start_time=data_fetcher.t_start)
        node.log(job.execution_report())
    node.results["ds_raw"] = dataset

## 8 · Analyse
Pure Python from here on: no hardware involved. The fit functions live in `calibration_utils/resonator_spectroscopy/analysis.py`;
the node records an **outcome** (`successful` / `failed`) per qubit, which the next step uses.

In [ ]:
@node.run_action(skip_if=node.parameters.simulate)
def analyse_data(node: QualibrationNode[Parameters, Quam]):
    node.results["ds_raw"] = process_raw_dataset(node.results["ds_raw"], node)
    node.results["ds_fit"], fit_results = fit_raw_data(node.results["ds_raw"], node)
    node.results["fit_results"] = {k: asdict(v) for k, v in fit_results.items()}
    log_fitted_results(node.results["fit_results"], log_callable=node.log)
    node.outcomes = {
        name: ("successful" if r["success"] else "failed") for name, r in node.results["fit_results"].items()
    }
    print(node.outcomes)

## 9 · Plot

In [ ]:
@node.run_action(skip_if=node.parameters.simulate)
def plot_data(node: QualibrationNode[Parameters, Quam]):
    fig_phase = plot_raw_phase(node.results["ds_raw"], node.namespace["qubits"])
    fig_amp = plot_raw_amplitude_with_fit(node.results["ds_raw"], node.namespace["qubits"], node.results["ds_fit"])
    plt.show()
    node.results["figures"] = {"phase": fig_phase, "amplitude": fig_amp}

## 10 · Update the state and save

<div style="background: rgba(46, 160, 67, 0.10); border-left: 4px solid rgba(46, 160, 67, 0.75); padding: 8px 14px; border-radius: 6px;">

🧩 **QUAlibrate** — `node.record_state_updates()` records every change made to the QUAM tree inside the block,
so it can be reviewed and applied. 🧩 **QUAM** — assigning `q.resonator.f_01 = ...` edits the machine state itself.
`node.save()` stores parameters, data, figures and the updates under `data/`.

</div>

The updated frequency is only written for qubits whose fit succeeded.

In [ ]:
@node.run_action(skip_if=node.parameters.simulate)
def update_state(node: QualibrationNode[Parameters, Quam]):
    with node.record_state_updates():
        for q in node.namespace["qubits"]:
            if node.outcomes[q.name] == "failed":
                continue
            q.resonator.f_01 = float(node.results["fit_results"][q.name]["frequency"])
            q.resonator.RF_frequency = float(node.results["fit_results"][q.name]["frequency"])


@node.run_action()
def save_results(node: QualibrationNode[Parameters, Quam]):
    node.save()

## Where next

<div style="background: rgba(128, 128, 128, 0.1); border-left: 4px solid rgba(128, 128, 128, 0.6); padding: 8px 14px; border-radius: 6px;">

* Browse `calibrations/` — every node follows this exact shape.
* Open **`01_workflow.ipynb`** to download the latest state and run several nodes in sequence.
* Docs: [QUA](https://docs.quantum-machines.co/latest/) · [QUAM](https://qua-platform.github.io/quam/) · [quam-builder](https://github.com/qua-platform/quam-builder) · [QUAlibrate](https://qua-platform.github.io/qualibrate/) · [qualibration-libs](https://github.com/qua-platform/qualibration-libs)

</div>